# 04 - Aggregate Results and Build Paper Artifacts

[Open in Google Colab](https://colab.research.google.com/github/zhaoqyu/Lab-NLP/blob/mike/alignment_benchmark/notebooks/04_analyze_results.ipynb)

**Objective.** Enforce experiment completeness, compute registered statistics, and export final tables and figures.

Use a GPU runtime. Persistent artifacts are written to Google Drive, so interrupted Colab sessions can resume. Run cells from top to bottom.

In [ ]:
import os
import subprocess
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
assert subprocess.run(['nvidia-smi'], check=False).returncode == 0, 'Select a GPU runtime first.'


In [ ]:
REPO_URL = 'https://github.com/zhaoqyu/Lab-NLP.git'
BRANCH = 'mike'
REPO_DIR = Path('/content/Lab-NLP')

if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, check=True)

os.chdir(REPO_DIR)
print('Repository:', REPO_DIR)


In [ ]:
subprocess.run(
    ['python', '-m', 'pip', 'install', '-q', '-r', 'alignment_benchmark/requirements-colab.txt'],
    check=True,
)
subprocess.run(
    ['python', '-m', 'pip', 'install', '-q', '-e', './alignment_benchmark', '--no-deps'],
    check=True,
)


In [ ]:
OUTPUT_ROOT = Path('/content/drive/MyDrive/Lab-NLP/valuebench-paper')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['VALUEBENCH_OUTPUT_ROOT'] = str(OUTPUT_ROOT)
CONFIG = 'alignment_benchmark/configs/paper.yaml'

def run(*arguments: str) -> None:
    command = ['valuebench', *arguments, '--config', CONFIG]
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)

print('Persistent output:', OUTPUT_ROOT)


In [ ]:
run('doctor', '--strict')


## 1. Completeness gate

Final aggregation deliberately fails if any registered construction or evaluation is missing.

In [ ]:
run('status')


## 2. Bootstrap registered metrics

In [ ]:
run('aggregate-results')


## 3. Export paper tables, figures, and checksums

In [ ]:
run('paper-artifacts')


In [ ]:
import pandas as pd
from IPython.display import display

paper = OUTPUT_ROOT / 'paper'
display(pd.read_csv(paper / 'table_main_method_comparison.csv'))
display(pd.read_csv(paper / 'table_per_value_results.csv').head(20))


In [ ]:
from IPython.display import Image, display

for image_path in sorted(paper.glob('figure_*.png')):
    print(image_path.name)
    display(Image(filename=str(image_path), width=900))


## 4. Create a portable paper-artifact archive

In [ ]:
import shutil

archive = shutil.make_archive(str(OUTPUT_ROOT / 'paper_artifacts'), 'zip', root_dir=paper)
print('Archive:', archive)


## Interpretation note

Do not write conclusions from direction alone. Use confidence intervals, FDR-adjusted per-value tests, sparse-cell flags, drift, seed stability, and efficiency together.